# 61 — Verbal aggression: the irtree workflow (paper Section 6.1)

This notebook is the code of Section 6.1. It fits De Boeck and Partchev's (2012) linear tree to the verbal aggression data, step by step:

1. read the data and declare the tree,
2. prepare the data with person and item covariates,
3. choose the node models and look at the priors,
4. fit, check convergence, interpret,
5. convert to the logistic metric, and
6. extend the nodes from Rasch to 2PL and compare the two models.

**Data.** `verbagg_long.csv` is lme4's `VerbAgg`: 316 persons, 24 items (*"A bus fails to stop for me. I would want to curse"*), responses no / perhaps / yes, with person covariates (Anger score, gender) and item covariates (behavior type: curse, scold, shout; situation: other or self to blame; mode: want or do).

**Runtime.** Each fit is timed; the last cell saves the table to `results/runtime_61.csv`.

From DeBoeck (2012):

The choice for perhaps" or yes" vs. no" which refers to the top node is about admitting
one may show an aggressive reaction. 

Admission is different from affirmation. The choice for yes" vs. "perhaps" refers to the second node and implies the affirmation of an aggressive
reaction. Admission and affirmation may be two qualitatively different responses and rely on
different latent traits. Affirmation is not necessarily more of the same or just stronger than
admission.

From Jeon & Deboeck, 2016
Recall
that in Fig. 2 the ‘Perhaps’ category is differentiated in Stage
1 instead of in Stage 2. The tree in Fig. 2 is useful for finding
out whether the ‘Perhaps’ response stems from a preliminary
feeling of certainty. The tree in Fig. 3 is more useful
for testing whether the ‘Perhaps’ response is a way to admit
verbal aggression. Based on the model fit, the tree in Fig. 3
(log-likelihood = −6087.22) appears to be a better representation
of the item response process for the data than the
tree in Fig. 2 (log-likelihood=−6188.76).


Based on the tree in Fig. 3, we fit a two-dimensional
item response tree model. The correlation between the two
dimensions is estimated as 0.38. This indicates that the two
decision processes (direction and certainty) are only weakly
correlated, implying that the resulting outcomes of the two
internal processes are not closely related to each other. That
is, the three response categories (‘No’, ‘Perhaps’,‘Yes’) may
not be really ordinal. The response scale seems to be multidimensional,
meaning that a different latent variable is
measured depending on the response option chosen by the
person (also per item).


1) Model 1: a reduced two-dimensional tree model that
includes node-specific main effects parameters (but
not node-specific item slopes and intercepts),

(2) Model 2: a unidimensional tree model that assumes
a single latent variable (but with node-specific item
slopes and intercepts), and


(3) Model 3: a bifactor tree model that assumes a general
latent trait for all response categories in addition to the
latent traits for the two internal nodes. ------------------------->>> This is not yet possible in the package



These results suggest that inference and decision
based on subjects’ latent trait scores can be biased
if non-ordinality of a response scale is ignored in data
analysis.





## Setup

In [1]:
# Install the package from this repository (editable), once per environment.
# The notebook lives in replication/<folder>/, so the repository root is two levels up.
import sys, pathlib
ROOT = pathlib.Path.cwd().parents[1]
!{sys.executable} -m pip install -q -e "{ROOT}[full]"

In [2]:
import warnings
import numpy as np, pandas as pd
import irtree
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore", category=FutureWarning)

RESULTS = pathlib.Path("results"); RESULTS.mkdir(exist_ok=True)

# Runtime log: wrap every expensive step in `with timed("label"):`.
# The last cell prints the table and saves it to results/.
import time, contextlib
RUNTIME = []

@contextlib.contextmanager
def timed(step):
    t0 = time.perf_counter()
    yield
    dt = time.perf_counter() - t0
    RUNTIME.append({"step": step, "seconds": round(dt, 1)})
    print(f"[{step}] " + (f"{dt / 60:.1f} min" if dt >= 60 else f"{dt:.1f} s"))

## Settings
`N_ITER` counts warm-up and retained draws; with the default evidence-based warm-up, the retained draws are reported by the fit.

In [3]:
N_ITER   = 4000
N_CHAINS = 4 # Some runs will be one chain and some 4 
SEED     = 2026
print(f"N_ITER={N_ITER}  N_CHAINS={N_CHAINS}  SEED={SEED}")

N_ITER=4000  N_CHAINS=4  SEED=2026


## Step 1: Declare the tree
The linear tree has two nodes. Node 1 separates *no* from *perhaps or yes*; node 2, reached only by *perhaps or yes*, separates *perhaps* from *yes*. Rows of the mapping matrix are the categories 0, 1, 2; `NaN` marks a node that is not on the path.

In [47]:
spec = irtree.IRTreeSpec(mapping=np.array([[0, np.nan],    # no
                                           [1, 0],         # perhaps
                                           [1, 1]]),       # yes
                         category_names=("no", "perhaps", "yes"),
                         node_names=("respond", "yes"))
spec

IRTreeSpec(K=3, R=2, latent_dims=2)
  Categories:        K = 3
  Nodes:             R = 2
  Latent dimensions: 2
  Outcomes per node: [2, 2]

## Step 2: Prepare the data
`load_data` in single-table mode: the long table carries the responses and both kinds of covariates; `roles` says which columns are person (X) and item (Q) covariates. It aligns everything by person and item id and expands the responses into the pseudo-item responses of each node.

### Step 2.1: Read the data
One row per person × item. The responses are recoded to 0 (no), 1 (perhaps), 2 (yes).

In [48]:
va = pd.read_csv("verbagg_long.csv")
va["response"] = va["resp"].map({"no": 0, "perhaps": 1, "yes": 2})
print(va.shape)
print(va["resp"].value_counts())
va.head()

(7584, 9)
resp
no         3973
perhaps    2081
yes        1530
Name: count, dtype: int64


,id,item,resp,Anger,Gender,btype,situ,mode,response
0,1,S1WantCurse,no,20,M,curse,other,want,0
1,2,S1WantCurse,no,11,M,curse,other,want,0
2,3,S1WantCurse,perhaps,17,F,curse,other,want,1
3,4,S1WantCurse,perhaps,21,F,curse,other,want,1
4,5,S1WantCurse,perhaps,17,F,curse,other,want,1


In [23]:
va[va["id"]==1]

,id,item,resp,Anger,Gender,btype,situ,mode,response
0,1,S1WantCurse,no,20,M,curse,other,want,0
316,1,S1WantScold,no,20,M,scold,other,want,0
632,1,S1WantShout,no,20,M,shout,other,want,0
948,1,S2WantCurse,no,20,M,curse,other,want,0
1264,1,S2WantScold,no,20,M,scold,other,want,0
1580,1,S2WantShout,no,20,M,shout,other,want,0
1896,1,S3WantCurse,no,20,M,curse,self,want,0
2212,1,S3WantScold,no,20,M,scold,self,want,0
2528,1,S3WantShout,perhaps,20,M,shout,self,want,1
2844,1,S4wantCurse,yes,20,M,curse,self,want,2


### Step 3.1: Covariates
All covariates are centered (Section 4: the traits and locations have fixed means, so uncentered covariates would shift them).

* person: `anger` (Anger score) and `male` (gender indicator);
* item: `scold`, `shout` (curse is the reference), `self` (self to blame), `do` (doing rather than wanting), each a centered indicator, plus `intercept`, a column of ones. Because the item locations have prior mean 0, the intercept's coefficient is the node's average location shift: the node intercept.

Notes: 
Generally, one should center the predictors. why? Because we are not adding an intercept in the model. If we were, then there would be a baseline constant person parameter beyond the explained part of the person latent variables and the residual that is centered around zERO. 

When the the person parameters are not centered, the intercept (same value across people) gets into the item parameter (if fixed). But if not fixed effect the extent of what it biases depends on the number of persons vs number of items. With generally less items

In [46]:
pers = va.drop_duplicates("id")
items = va.drop_duplicates("item")

person_cov = pd.DataFrame({"id": pers["id"],
                           "anger": pers["Anger"],
                           "male":  pers["Gender"] == "M"})


item_cov = pd.DataFrame({"item": items["item"],
                         "intercept": 1.0,
                         "scold": items["btype"] == "scold",
                         "shout": items["btype"] == "shout",
                         "self": items["situ"] == "self",
                         "do":    items["mode"] == "do"})

table = va[["id", "item", "response"]].merge(person_cov, on="id").merge(item_cov, on="item")

X_NAMES = ["anger", "male"]

Q_NAMES = ["intercept", "scold", "shout", "self", "do"]

table.head()

,id,item,response,anger,male,intercept,scold,shout,self,do
0,1,S1WantCurse,0,20,True,1.0,False,False,False,False
1,2,S1WantCurse,0,11,True,1.0,False,False,False,False
2,3,S1WantCurse,1,17,False,1.0,False,False,False,False
3,4,S1WantCurse,1,21,False,1.0,False,False,False,False
4,5,S1WantCurse,1,17,False,1.0,False,False,False,False


### 2.3 : Load the data

In [31]:
data = irtree.load_data(table, spec=spec, person_id="id", item_id="item",
                        response_col="response", roles={"X": X_NAMES, "Q": Q_NAMES},
                        categories_start=0, center = True, scale = True)


print(irtree.summarize_prepared(data, spec))

Ingest: P=316 persons, I=24 items (first-seen order).
  X:  Scaled: ['anger', 'male']
  Q:  Scaled: ['scold', 'shout', 'self', 'do']; unchanged: ['intercept']
PreparedData: P=316, I=24, R=2, missing=0
  Node 1: n_relevant=7584, mean_Y*=0.476
  Node 2: n_relevant=3611, mean_Y*=0.424
  X (person covariates): K_x=2, n_missing=0
  Q (item covariates): K_q=5, n_missing=0


In [32]:
data1 = irtree.load_data(table, spec=spec, person_id="id", item_id="item",
                        response_col="response",
                        categories_start=0, center = False, scale = True)

Ingest: P=316 persons, I=24 items (first-seen order).


## Step 3: Node models and priors 

Four node models are specified here. Two use a probit link and the remaining two use logit links at both nodes
For each node type, the model is ran with/without covariates.


### Step 3.1: Node models & priors probit links

In [51]:
# with covariates
nodes_rasch_cov = [irtree.ProbitRaschNode(person_covariates=True, item_covariates=True),
               irtree.ProbitRaschNode(person_covariates=True, item_covariates=True)] # node model
priors_rasch_cov = irtree.build_default_priors(spec, nodes_rasch_cov) # priors

# without covariates
nodes_rasch_no_cov = [irtree.ProbitRaschNode(),
               irtree.ProbitRaschNode()] # node model
priors_rasch_no_cov = irtree.build_default_priors(spec, nodes_rasch_no_cov) # priors

### Step 3.2: Node models & priors logit links

In [53]:
# with covariates
nodes_log_rasch_cov = [irtree.LogitRaschNode(person_covariates=True, item_covariates=True),
               irtree.LogitRaschNode(person_covariates=True, item_covariates=True)] # node model

# without covariates
nodes_log_rasch_no_cov = [irtree.LogitRaschNode(),
               irtree.LogitRaschNode()]

## Step 4: Fit

In [35]:
# probit rasch with covariates
with timed("fit probit Rasch tree with covariates"):
    fit_rasch_ = irtree.irtree_fit(data, spec, nodes_rasch_cov, priors=priors_rasch_cov,
                                  n_iter=N_ITER, n_chains=1, seed=SEED, progress=True)

sample 400/4000 (10%) · 216.3 it/s · elapsed 0:01 · eta 0:16 · loglik -7053.9 · accept 0.40
sample 800/4000 (20%) · 213.6 it/s · elapsed 0:03 · eta 0:14 · loglik -5601.7 · accept 0.40
sample 1200/4000 (30%) · 209.1 it/s · elapsed 0:05 · eta 0:13 · loglik -5609.3 · accept 0.40
sample 1600/4000 (40%) · 206.0 it/s · elapsed 0:07 · eta 0:11 · loglik -5589.6 · accept 0.40
sample 2000/4000 (50%) · 203.9 it/s · elapsed 0:09 · eta 0:09 · loglik -5614.7 · accept 0.40
sample 2400/4000 (60%) · 201.7 it/s · elapsed 0:11 · eta 0:07 · loglik -5622.9 · accept 0.39
sample 2800/4000 (70%) · 200.6 it/s · elapsed 0:13 · eta 0:05 · loglik -5584.6 · accept 0.40
sample 3200/4000 (80%) · 200.4 it/s · elapsed 0:15 · eta 0:03 · loglik -5586.0 · accept 0.40
sample 3600/4000 (90%) · 200.3 it/s · elapsed 0:17 · eta 0:01 · loglik -5610.4 · accept 0.40
sample 4000/4000 (100%) · 200.4 it/s · elapsed 0:19 · eta 0:00 · loglik -5613.5 · accept 0.40
done 4000/4000 (100%) · elapsed 0:19



irtree fit complete  (1 chain × 4000 iterations, 3500 posterior draws)
   parameters: 690 PASS · 13 WARN · 0 FAIL   (of 703)
   3 flagged only for under-warmup — resolve by discarding the first ~10% (recommended_burnin).
   (overall label is advisory; the per-parameter table is the decision. Call fit.diagnostics().)
   status key:
     PASS (converged): stationary with no burn-in needed, adequate mixing.
     WARN (usable, but check): at worst under-warmed (a recommended burn-in is given), OR the two stationarity tests (Geweke & Heidelberger-Welch) disagree (one passes, one flags), OR mixing/precision is marginal (low ESS, high MCSE, R-hat slightly > 1).
     FAIL (did not converge): fails BOTH stationarity tests (Geweke & Heidelberger-Welch), or R-hat exceeds its hard threshold. A parameter that passes EITHER stationarity test is a WARN, not a FAIL.
⚠  Single chain — R-hat unavailable. Stationarity via Geweke + Heidelberger-Welch.
✓ Stationarity (Geweke):   All Geweke within threshol

In [ ]:
# probit Rasch without covariates

with timed("fit probit Rasch tree without"):
    fit_rasch_no_cov = irtree.irtree_fit(data, spec, nodes_rasch_no_cov, priors=priors_rasch_no_cov,
                                  n_iter=N_ITER, n_chains=1, seed=SEED, progress=True)

In [36]:
with timed("fit Rasch tree"):
    fit_rasch_no_cov = irtree.irtree_fit(data, spec, nodes_rasch_no_cov, priors=priors_rasch_no_cov,
                                  n_iter=N_ITER, n_chains=1, seed=SEED, progress=True)

sample 400/4000 (10%) · 452.2 it/s · elapsed 0:00 · eta 0:07 · loglik -7053.9 · accept 0.41
sample 800/4000 (20%) · 416.3 it/s · elapsed 0:01 · eta 0:07 · loglik -5596.1 · accept 0.44
sample 1200/4000 (30%) · 399.6 it/s · elapsed 0:03 · eta 0:07 · loglik -5617.2 · accept 0.43
sample 1600/4000 (40%) · 397.2 it/s · elapsed 0:04 · eta 0:06 · loglik -5616.3 · accept 0.42
sample 2000/4000 (50%) · 391.8 it/s · elapsed 0:05 · eta 0:05 · loglik -5624.2 · accept 0.43
sample 2400/4000 (60%) · 390.7 it/s · elapsed 0:06 · eta 0:04 · loglik -5604.4 · accept 0.43
sample 2800/4000 (70%) · 390.1 it/s · elapsed 0:07 · eta 0:03 · loglik -5598.8 · accept 0.43
sample 3200/4000 (80%) · 390.1 it/s · elapsed 0:08 · eta 0:02 · loglik -5596.5 · accept 0.43
sample 3600/4000 (90%) · 389.1 it/s · elapsed 0:09 · eta 0:01 · loglik -5601.3 · accept 0.42
sample 4000/4000 (100%) · 386.9 it/s · elapsed 0:10 · eta 0:00 · loglik -5595.1 · accept 0.42
done 4000/4000 (100%) · elapsed 0:10



irtree fit complete  (1 chain × 4000 iterations, 3600 posterior draws)
   parameters: 676 PASS · 9 WARN · 0 FAIL   (of 685)
   3 flagged only for under-warmup — resolve by discarding the first ~10% (recommended_burnin).
   (overall label is advisory; the per-parameter table is the decision. Call fit.diagnostics().)
   status key:
     PASS (converged): stationary with no burn-in needed, adequate mixing.
     WARN (usable, but check): at worst under-warmed (a recommended burn-in is given), OR the two stationarity tests (Geweke & Heidelberger-Welch) disagree (one passes, one flags), OR mixing/precision is marginal (low ESS, high MCSE, R-hat slightly > 1).
     FAIL (did not converge): fails BOTH stationarity tests (Geweke & Heidelberger-Welch), or R-hat exceeds its hard threshold. A parameter that passes EITHER stationarity test is a WARN, not a FAIL.
⚠  Single chain — R-hat unavailable. Stationarity via Geweke + Heidelberger-Welch.
✓ Stationarity (Geweke):   All Geweke within threshold

## Step 7 — Convergence
The console summary classifies every parameter; `diagnostics()` returns the full table (R-hat, bulk and tail ESS, MCSE/SD).

In [42]:
rep = fit_rasch.diagnostics

In [43]:
rep_no_cov = fit_rasch.diagnostics()
rep_no_cov

,n,n_pass,n_warn,n_fail,max_rhat,min_ess_bulk,min_ess_tail,max_mcse_sd
kind,,,,,,,,
theta,632,621,11,0,,217,707,0.068
b,48,48,0,0,,1204,2536,0.029
sigma2,2,1,1,0,,248,1082,0.064
b.variance,2,2,0,0,,1810,1962,0.024
rho,1,0,1,0,,375,616,0.052
beta,4,4,0,0,,1520,2863,0.026
kappa,10,10,0,0,,1578,2749,0.025
beta.variance,2,2,0,0,,3321,3500,0.017
kappa.variance,2,2,0,0,,3367,3273,0.017


## Step 8 — Interpret
The structural parameters: trait variances (estimated on Rasch dimensions), the trait correlation, the item-variance of each node, and the covariate effects. Positive coefficients raise the probability of the branch coded 1 (responding at node 1, saying *yes* at node 2).

In [44]:
summ = fit_rasch.summary_df()
summ

,mean,sd,q2.5,q50,q97.5,hdi_lo,hdi_hi,n_eff
param,,,,,,,,
theta_1_1,-0.371874,0.267115,-0.896622,-0.375556,0.156755,-0.875667,0.172080,1642.798246
theta_1_2,-1.374755,0.381377,-2.143765,-1.367629,-0.646320,-2.155528,-0.669298,687.121541
theta_1_3,-0.076576,0.263832,-0.590406,-0.072373,0.442496,-0.568326,0.460430,1266.312966
theta_1_4,0.253284,0.263308,-0.260878,0.250246,0.777693,-0.275359,0.743896,1551.603648
theta_1_5,-0.110925,0.273462,-0.665971,-0.109493,0.416791,-0.615650,0.462179,1425.409554
...,...,...,...,...,...,...,...,...
kappa_2_do,-0.165807,0.089064,-0.345572,-0.164112,0.012971,-0.348961,0.006818,2547.641117
beta.variance_1,0.961584,1.369440,0.175441,0.601860,3.991445,0.098010,2.756211,3321.055609
beta.variance_2,1.050781,1.604228,0.191662,0.647967,4.365859,0.093195,3.050014,3456.690918


In [ ]:
summ = fit_rasch.summary_df()
summ

In [20]:
structural = [n for n in summ.index if n.startswith(("sigma2", "rho", "b.variance", "beta_", "kappa_"))]
summ.loc[structural, ["mean", "sd", "hdi_lo", "hdi_hi", "n_eff"]].round(3)

,mean,sd,hdi_lo,hdi_hi,n_eff
param,,,,,
sigma2_1,0.640,0.064,0.519,0.764,764.126
sigma2_2,0.585,0.075,0.440,0.730,247.813
b.variance_1,0.166,0.060,0.073,0.283,2239.975
b.variance_2,0.163,0.060,0.066,0.283,1809.548
rho_1_2,0.339,0.070,0.222,0.489,375.234
beta_1_anger,0.161,0.048,0.061,0.251,1831.350
beta_1_male,0.080,0.047,-0.006,0.182,2343.445
beta_2_anger,0.297,0.052,0.193,0.398,1520.345
beta_2_male,0.092,0.050,-0.003,0.192,2016.097


In [41]:
structural = [n for n in summ.index if n.startswith(("sigma2", "rho", "b.variance", "beta_", "kappa_"))]
summ.loc[structural, ["mean", "sd", "hdi_lo", "hdi_hi", "n_eff"]].round(3)

,mean,sd,hdi_lo,hdi_hi,n_eff
param,,,,,
sigma2_1,0.640,0.064,0.519,0.764,764.126
sigma2_2,0.585,0.075,0.440,0.730,247.813
b.variance_1,0.166,0.060,0.073,0.283,2239.975
b.variance_2,0.163,0.060,0.066,0.283,1809.548
rho_1_2,0.339,0.070,0.222,0.489,375.234
beta_1_anger,0.161,0.048,0.061,0.251,1831.350
beta_1_male,0.080,0.047,-0.006,0.182,2343.445
beta_2_anger,0.297,0.052,0.193,0.398,1520.345
beta_2_male,0.092,0.050,-0.003,0.192,2016.097


In [45]:
summ_no_cov = fit_rasch_no_cov.summary_df()
summ_no_cov

,mean,sd,q2.5,q50,q97.5,hdi_lo,hdi_hi,n_eff
param,,,,,,,,
theta_1_1,-0.236646,0.255136,-0.731134,-0.233750,0.276366,-0.690614,0.309100,1504.136790
theta_1_2,-1.491399,0.367046,-2.263355,-1.474023,-0.804830,-2.205693,-0.757520,714.896396
theta_1_3,-0.228959,0.258507,-0.739725,-0.227907,0.268495,-0.725047,0.281037,1524.697435
theta_1_4,0.223357,0.265293,-0.288555,0.223235,0.744299,-0.297450,0.733023,1311.864545
theta_1_5,-0.247347,0.275908,-0.772180,-0.243451,0.304962,-0.761776,0.313169,1525.439010
...,...,...,...,...,...,...,...,...
sigma2_1,0.665169,0.066319,0.540295,0.663116,0.800812,0.534143,0.794234,630.177057
sigma2_2,0.655048,0.084423,0.502891,0.651780,0.836495,0.491145,0.819469,499.449440
b.variance_1,0.524339,0.158730,0.296630,0.496144,0.915426,0.269671,0.847370,2747.608451


In [34]:
structural = [n for n in summ_no_cov.index if n.startswith(("sigma2", "rho", "b.variance"))]
summ_no_cov.loc[structural, ["mean", "sd", "hdi_lo", "hdi_hi", "n_eff"]].round(3) 

,mean,sd,hdi_lo,hdi_hi,n_eff
param,,,,,
sigma2_1,0.665,0.066,0.534,0.794,630.177
sigma2_2,0.655,0.084,0.491,0.819,499.449
b.variance_1,0.524,0.159,0.270,0.847,2747.608
b.variance_2,0.535,0.179,0.257,0.908,1810.484
rho_1_2,0.361,0.067,0.228,0.490,281.187


## Step 9 — The logistic metric
`to_logit()` puts the summary on the logistic-equivalent scale (D = 1.702) so the estimates can be read against logistic results such as those of Section 7.2. On these slope-free dimensions the factor goes on the traits, locations and coefficients; variances take its square; the correlation is unchanged. The conversion is approximate in the tails.

In [35]:
lg = fit_rasch.to_logit()
print(*lg.attrs.get("notes", []), sep="\n")
lg.loc[structural, ["mean", "sd", "factor"]].round(3)

,mean,sd,factor
param,,,
sigma2_1,1.853,0.184,2.897
sigma2_2,1.696,0.218,2.897
b.variance_1,0.481,0.173,2.897
b.variance_2,0.473,0.175,2.897
rho_1_2,0.339,0.070,1.000


## Run the logistic versions

In [37]:
nodes_log_rasch = [irtree.LogitRaschNode(person_covariates=True, item_covariates=True),
               irtree.LogitRaschNode(person_covariates=True, item_covariates=True)]

nodes_log_rasch_no_cov = [irtree.LogitRaschNode(),
               irtree.LogitRaschNode()]


with timed("fit Rasch tree"):
    fit_rasch_log = irtree.irtree_fit(data, spec, nodes_log_rasch, priors=priors_rasch,
                                  n_iter=N_ITER, n_chains=1, seed=SEED, progress=True)

with timed("fit Rasch tree"):
    fit_rasch_log_no_cov = irtree.irtree_fit(data, spec, nodes_log_rasch_no_cov, priors=priors_rasch_no_cov,
                              n_iter=N_ITER, n_chains=1, seed=SEED, progress=True)

sample 400/4000 (10%) · 114.1 it/s · elapsed 0:03 · eta 0:31 · loglik -5622.6 · accept 0.39
sample 800/4000 (20%) · 114.4 it/s · elapsed 0:06 · eta 0:27 · loglik -5629.7 · accept 0.39
sample 1200/4000 (30%) · 114.5 it/s · elapsed 0:10 · eta 0:24 · loglik -5596.6 · accept 0.38
sample 1600/4000 (40%) · 114.5 it/s · elapsed 0:13 · eta 0:20 · loglik -5599.6 · accept 0.38
sample 2000/4000 (50%) · 114.6 it/s · elapsed 0:17 · eta 0:17 · loglik -5612.0 · accept 0.38
sample 2400/4000 (60%) · 114.2 it/s · elapsed 0:21 · eta 0:14 · loglik -5609.7 · accept 0.38
sample 2800/4000 (70%) · 114.2 it/s · elapsed 0:24 · eta 0:10 · loglik -5612.7 · accept 0.38
sample 3200/4000 (80%) · 114.4 it/s · elapsed 0:27 · eta 0:06 · loglik -5659.2 · accept 0.38
sample 3600/4000 (90%) · 114.5 it/s · elapsed 0:31 · eta 0:03 · loglik -5636.5 · accept 0.38
sample 4000/4000 (100%) · 114.5 it/s · elapsed 0:34 · eta 0:00 · loglik -5605.5 · accept 0.38
done 4000/4000 (100%) · elapsed 0:34



irtree fit complete  (1 chain × 4000 iterations, 3600 posterior draws)
   parameters: 693 PASS · 10 WARN · 0 FAIL   (of 703)
   4 flagged only for under-warmup — resolve by discarding the first ~10% (recommended_burnin).
   (overall label is advisory; the per-parameter table is the decision. Call fit.diagnostics().)
   status key:
     PASS (converged): stationary with no burn-in needed, adequate mixing.
     WARN (usable, but check): at worst under-warmed (a recommended burn-in is given), OR the two stationarity tests (Geweke & Heidelberger-Welch) disagree (one passes, one flags), OR mixing/precision is marginal (low ESS, high MCSE, R-hat slightly > 1).
     FAIL (did not converge): fails BOTH stationarity tests (Geweke & Heidelberger-Welch), or R-hat exceeds its hard threshold. A parameter that passes EITHER stationarity test is a WARN, not a FAIL.
⚠  Single chain — R-hat unavailable. Stationarity via Geweke + Heidelberger-Welch.
✓ Stationarity (Geweke):   All Geweke within threshol

sample 400/4000 (10%) · 223.0 it/s · elapsed 0:01 · eta 0:16 · loglik -5633.6 · accept 0.45
sample 800/4000 (20%) · 224.3 it/s · elapsed 0:03 · eta 0:14 · loglik -5578.4 · accept 0.44
sample 1200/4000 (30%) · 224.7 it/s · elapsed 0:05 · eta 0:12 · loglik -5618.1 · accept 0.44
sample 1600/4000 (40%) · 225.0 it/s · elapsed 0:07 · eta 0:10 · loglik -5612.8 · accept 0.44
sample 2000/4000 (50%) · 222.8 it/s · elapsed 0:08 · eta 0:08 · loglik -5608.2 · accept 0.44
sample 2400/4000 (60%) · 223.0 it/s · elapsed 0:10 · eta 0:07 · loglik -5651.7 · accept 0.44
sample 2800/4000 (70%) · 223.3 it/s · elapsed 0:12 · eta 0:05 · loglik -5613.7 · accept 0.44
sample 3200/4000 (80%) · 222.1 it/s · elapsed 0:14 · eta 0:03 · loglik -5599.6 · accept 0.44
sample 3600/4000 (90%) · 222.3 it/s · elapsed 0:16 · eta 0:01 · loglik -5603.2 · accept 0.44
sample 4000/4000 (100%) · 222.1 it/s · elapsed 0:18 · eta 0:00 · loglik -5617.0 · accept 0.44
done 4000/4000 (100%) · elapsed 0:18



irtree fit complete  (1 chain × 4000 iterations, 3400 posterior draws)
   parameters: 671 PASS · 14 WARN · 0 FAIL   (of 685)
   5 flagged only for under-warmup — resolve by discarding the first ~10% (recommended_burnin).
   (overall label is advisory; the per-parameter table is the decision. Call fit.diagnostics().)
   status key:
     PASS (converged): stationary with no burn-in needed, adequate mixing.
     WARN (usable, but check): at worst under-warmed (a recommended burn-in is given), OR the two stationarity tests (Geweke & Heidelberger-Welch) disagree (one passes, one flags), OR mixing/precision is marginal (low ESS, high MCSE, R-hat slightly > 1).
     FAIL (did not converge): fails BOTH stationarity tests (Geweke & Heidelberger-Welch), or R-hat exceeds its hard threshold. A parameter that passes EITHER stationarity test is a WARN, not a FAIL.
⚠  Single chain — R-hat unavailable. Stationarity via Geweke + Heidelberger-Welch.
✓ Stationarity (Geweke):   All Geweke within threshol

In [39]:
summ_log = fit_rasch_log.summary_df()
summ_log_no_cov = fit_rasch_log_no_cov.summary_df()




structural = [n for n in summ_log_no_cov.index if n.startswith(("sigma2", "rho", "b.variance"))]
summ_log_no_cov.loc[structural, ["mean", "sd", "hdi_lo", "hdi_hi", "n_eff"]].round(3) 

,mean,sd,hdi_lo,hdi_hi,n_eff
param,,,,,
sigma2_1,1.937,0.200,1.534,2.317,632.328
sigma2_2,1.860,0.239,1.416,2.348,258.829
b.variance_1,1.375,0.422,0.650,2.191,2542.223
b.variance_2,1.352,0.462,0.586,2.246,1509.938
rho_1_2,0.345,0.068,0.209,0.473,302.824


In [40]:
structural = [n for n in summ_log.index if n.startswith(("sigma2", "rho", "b.variance"))]
summ_log.loc[structural, ["mean", "sd", "hdi_lo", "hdi_hi", "n_eff"]].round(3) 

,mean,sd,hdi_lo,hdi_hi,n_eff
param,,,,,
sigma2_1,1.857,0.191,1.490,2.235,593.317
sigma2_2,1.621,0.217,1.210,2.033,244.368
b.variance_1,0.275,0.101,0.115,0.470,1643.514
b.variance_2,0.245,0.096,0.101,0.429,935.595
rho_1_2,0.338,0.067,0.205,0.465,210.670


## Step 10 — From Rasch to 2PL nodes
Replacing the node models is the only change. On these dimensions every node now has slopes, so the trait variances are fixed at 1 (identification, Section 4) and the slopes carry the scale. The correlation is scale-free and stays estimated.

In [ ]:
nodes_2pl = [irtree.ProbitTwoPLNode(person_covariates=True, item_covariates=True),
             irtree.ProbitTwoPLNode(person_covariates=True, item_covariates=True)]
priors_2pl = irtree.build_default_priors(spec, nodes_2pl)
priors_2pl.set("beta",  irtree.NormalPrior(mean=0.0, variance=10.0))
priors_2pl.set("kappa", irtree.NormalPrior(mean=0.0, variance=10.0))
with timed("fit 2PL tree"):
    fit_2pl = irtree.irtree_fit(data, spec, nodes_2pl, priors=priors_2pl,
                                n_iter=N_ITER, n_chains=N_CHAINS, seed=SEED, progress=False)

In [ ]:
summ2 = fit_2pl.summary_df()
slopes = summ2.loc[[n for n in summ2.index if n.startswith("a_")], "mean"]
print("slopes, node 1:", slopes.filter(like="a_1_").round(2).to_numpy())
print("slopes, node 2:", slopes.filter(like="a_2_").round(2).to_numpy())
pd.DataFrame({"Rasch": summ.loc["rho_1_2", ["mean", "sd"]],
              "2PL":   summ2.loc["rho_1_2", ["mean", "sd"]]}).round(3)

## Step 11 — Compare the two trees
Approximate leave-one-out cross-validation (PSIS-LOO), leaving out one person at a time. `compare_loo(first, second)` returns `delta_elpd` = elpd(second) − elpd(first): here positive favours the 2PL tree. Read it against `se_delta`.

In [ ]:
with timed("LOO both models"):
    loo_rasch, loo_2pl = fit_rasch.loo(), fit_2pl.loo()
print("Rasch:", loo_rasch)
print("2PL:  ", loo_2pl)
irtree.compare_loo(loo_rasch, loo_2pl)

## Step 12 — Report (optional)
A self-contained HTML file with the convergence table and trace plots (not committed: `irtree_report*.html` is git-ignored).

In [ ]:
with timed("HTML report"):
    fit_2pl.report("irtree_report_61_2pl.html")

## Runtime

In [ ]:
rt = pd.DataFrame(RUNTIME)
rt.loc[len(rt)] = {"step": "TOTAL", "seconds": rt["seconds"].sum()}
rt.to_csv(RESULTS / "runtime_61.csv", index=False)
rt